# Data exploration

Thin runner over `iiss.data.eda`, `iiss.data.validate` and `iiss.analysis.viz`. All logic lives in `src/iiss/`; this notebook only sets parameters, calls functions and shows the results.

- Tables and figures are written to `results/eda/` (same files as `python -m iiss.data.eda`).
- Images are loaded with `iiss.data.images.load_image`, so the 4 rotation-fixed images line up with their masks.
- Runs on CPU in about a minute. `SMOKE = True` only reduces the number of sample images.

## Parameters

In [ ]:
SPLITS = ['train', 'valid', 'test']
N_SAMPLES = 6          # sample images per split in the gallery
SEED = 42
SAVE_GALLERIES = True  # also write results/eda/samples_<split>.jpg
SMOKE = False          # True: 3 samples per split, no saved galleries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from iiss.analysis.viz import sample_gallery
from iiss.data.coco import load_split
from iiss.data.eda import run_eda
from iiss.data.images import default_rotation_fixes
from iiss.utils.config import load_yaml
from iiss.utils.log import setup_logging
from iiss.utils.paths import repo_root

setup_logging()
if SMOKE:
    N_SAMPLES, SAVE_GALLERIES = 3, False
ROOT = repo_root()
BASE = load_yaml(ROOT / 'configs' / 'base.yaml')
EDA_DIR = ROOT / BASE['paths']['results_root'] / 'eda'
FIXES = default_rotation_fixes()
pd.set_option('display.max_columns', 20)

## Dataset validation status

From `results/data_validation.md` (regenerate with `python -m iiss.data.validate`).

In [ ]:
report = (ROOT / BASE['paths']['results_root'] / 'data_validation.md').read_text(encoding='utf-8')
display(Markdown('\n'.join(l for l in report.splitlines() if l.startswith('- '))))

## Run the EDA

In [ ]:
files = run_eda(BASE['paths']['dataset_root'], SPLITS, EDA_DIR)
sorted(p.name for p in files.values())

## Image sizes and resize direction per input size S

In [ ]:
display(pd.read_csv(files['long_side_stats']))
display(Image(filename=files['long_side_png'], width=650))

## Instances per image

In [ ]:
display(pd.read_csv(files['instances_per_image']))
display(Image(filename=files['ipi_png'], width=650))

## Object sizes at native resolution and after resizing to S

COCO bins: small < 32², medium < 96², large ≥ 96² px².

In [ ]:
display(pd.read_csv(files['object_sizes']))
display(Image(filename=files['small_png'], width=550))

## Class counts per split

In [ ]:
display(Image(filename=files['class_counts_png'], width=700))

## Train vs test class shift

Sorted by absolute change in instance share (percentage points).

In [ ]:
shift = pd.read_csv(files['class_shift'])
display(shift[shift['status'] != 'present_in_both'])
display(shift[shift['status'] == 'present_in_both'].head(15))

## Sample images with ground-truth masks

In [ ]:
for name in SPLITS:
    split = load_split(BASE['paths']['dataset_root'], name)
    fig = sample_gallery(split, n=N_SAMPLES, seed=SEED, fixes=FIXES)
    if SAVE_GALLERIES:
        fig.savefig(EDA_DIR / f'samples_{name}.jpg', dpi=80, pil_kwargs={'quality': 85})
    display(fig)
    plt.close(fig)

## Rotation-fixed images

These 4 files are stored rotated by 90° relative to their annotations; `load_image` rotates them back (`data.rotation_fixes`).

In [ ]:
for name in ['train', 'valid']:
    split = load_split(BASE['paths']['dataset_root'], name)
    fixed = [i['file_name'] for i in split.images if i['file_name'] in FIXES]
    fig = sample_gallery(split, file_names=fixed, fixes=FIXES, cols=2)
    display(fig)
    plt.close(fig)